**This notebook is used to load a file data as it is into the table**

In [0]:
%run "/Workspace/Users/wade.amit8895@outlook.com/Azure-Databricks-Projects/customer_billing/notebooks/utility/common_functions"

In [0]:
dbutils.widgets.text("file_name", "")
dbutils.widgets.text("pipeline_name", "customer_billing")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("unzip_path", "")

file_name = dbutils.widgets.get("file_name")
pipeline_name = dbutils.widgets.get("pipeline_name")
run_id = dbutils.widgets.get("run_id")
unzip_path = dbutils.widgets.get("unzip_path")

run_start_time = datetime.now()
task_name = "stage_2_load_data_to_bronze"

if not run_id:
    raise ValueError("run_id parameter is required")

In [0]:
final_unzip_path = unzip_path
file_name_prefix = "summary"
summary_file_name, summary_file_path = read_recursive_file_data(final_unzip_path)
print("summary_file_name =>", summary_file_name)
print("summary_file_path =>", summary_file_path)

In [0]:
#AW
# Calling unzip function with email notification
all_files = dbutils.fs.ls(bronze_all_files)
for actual_file_path in all_files:
    absolute_path = bronze_yaml_path + actual_file_path.name
    table_name = actual_file_path.name.split(".")[0]
    try:
        # Read YAML configuration
        reading_yaml_data = open_file_read_data(absolute_path)
        ytable_name = reading_yaml_data["table"]["name"]
        ytable_schema_name = reading_yaml_data["table"]["schema_name"]
        final_table_name = ytable_schema_name + "." + ytable_name
        ytable_source_file = reading_yaml_data["table"]["source_file"]
        ytable_columns_names = reading_yaml_data["table"]["columns"]

        # File name pattern
        file_name_prefix = ytable_source_file.split("*")[0]
        file_name_pattern = rf"^{file_name_prefix}_(\d{{8}})_(\d{{6}})\.csv$"

        # Recursive search
        final_file_name, final_file_path = read_recursive_file_data(final_unzip_path)
        print("final_file_name =>", final_file_name)
        print("final_file_path =>", final_file_path)

        # START LOG: CSV filename is now available
        process_log("Start", table_name=table_name, layer="bronze", file_name=final_file_name)

        # File name validation
        file_name_validation = match_file_name_pattern(file_name_pattern, final_file_name)
        print("file_name_validation =>", file_name_validation)

        # Data load process
        df_summary = (spark.read.format("csv").option("header", True).load(summary_file_path))
        df_column = df_summary.columns
        df_summary = df_summary.toDF(*[c.strip() for c in df_column])
        final_file_name_without_ext = final_file_name.replace(".csv", "")
        summary_count = (df_summary.filter(col("file_name").contains(final_file_name_without_ext)).select("count").first()[0])
        df_final_file_name = (spark.read.format("csv").option("header", True).load(final_file_path))
        table_count = df_final_file_name.count()
        if table_count == (int(summary_count) - 1):
            df_final_file_name.write.format("delta").mode("overwrite").saveAsTable(final_table_name)
            data_load_message = "Data load successfully"
            print("data_load_message => ", data_load_message)
            # Get total records inserted
            latest_history = (spark.sql(f"DESCRIBE HISTORY bronze.{table_name}").select("operationMetrics").first())
            if latest_history:
                op_metrics = latest_history["operationMetrics"]
                total_rows = int(op_metrics.get("numOutputRows", 0))
            else:
                total_rows = 0
            print("total_inserted_count =>", total_rows)
            process_log("Success", table_name=table_name, layer="bronze", total_inserted_count=total_rows, file_name=final_file_name)
        else:
            data_load_message = "Summary count is not matched hence process stopped"
            print("data_load_message => ", data_load_message)
            raise
        # spark.sql("""SELECT * FROM control.etl_run_log ORDER BY run_end_time DESC LIMIT 1""").show()
    except Exception as e:
        process_log("Failed", table_name=table_name, layer="bronze", error_message=str(e))
        # send_mail_fn(f"{pipeline_name}-{task_name}-{table_name}- Bronze layer is failed", str(e))
        raise